# MetaTrader 5 - Historical Candlestick Chart Analysis

This notebook demonstrates how to connect to the **MetaTrader 5 (MT5)** trading terminal, fetch historical candlestick data (rates) for a specific target date from 2-5 years ago, and visualize it on three different timeframes: **H1**, **M5**, and **M1**.

### Timeframe Specifications:
1. **H1 (1 Hour)**: Shows the day before, the target day, and the day after (a 3-day window) to provide broader market context.
2. **M5 (5 Minutes)**: Shows intraday price action for the target date.
3. **M1 (1 Minute)**: Shows highly detailed intraday price action for the target date.

In [ ]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
from datetime import datetime, timedelta

# Set pandas options for better display
pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 1000)

In [ ]:
# Initialize MetaTrader 5 connection
if not mt5.initialize():
    print("MetaTrader5 initialization failed, error code:", mt5.last_error())
    quit()
else:
    print("MetaTrader5 initialized successfully.")

    # Print terminal connection info
    info = mt5.terminal_info()
    print(f"Connected to: {info.company} - {info.name}")
    print(f"Version: {mt5.version()}")

## Define Target Date and Symbol

Set the `SYMBOL` and the historical `TARGET_DATE` (e.g. 2-5 years ago) that you want to analyze.

**Note**: Forex markets are closed on weekends (Saturdays and Sundays). If you select a weekend date, the fetched data will be empty or incomplete.

In [ ]:
# --- Configuration ---
SYMBOL = "GOOGL"          # Symbol to fetch (e.g., EURUSD, USDJPY, XAUUSD)
TARGET_DATE = "2026-06-30"  # Target date in YYYY-MM-DD format (approx 5 years ago)

## Fetch and Filter Historical Data

We request candle data (rates) from MetaTrader 5 using `mt5.copy_rates_range`.

To handle differences between the MT5 terminal server's timezone and local time, we request a slightly wider window (\u00b12 days) and then filter the retrieved candles precisely to the calendar day boundaries using pandas.

In [ ]:
# Convert configuration to datetime objects
target_datetime = datetime.strptime(TARGET_DATE, "%Y-%m-%d")

# Helper to check if a date is a weekend
def is_weekend(date_obj):
    return date_obj.weekday() >= 5  # 5: Saturday, 6: Sunday

if is_weekend(target_datetime):
    print(f"WARNING: The target date {TARGET_DATE} is a weekend ({target_datetime.strftime('%A')}).")
    print("Forex markets are closed on weekends. Charts may be empty. Please select a weekday for full data.")

# Calculate local calendar ranges
h1_start_local = target_datetime - timedelta(days=1)
h1_end_local = target_datetime + timedelta(days=1)

# Wider request bounds to guarantee we capture all candles across server timezone shifts
request_start = target_datetime - timedelta(days=2)
request_end = target_datetime + timedelta(days=3)

print(f"Fetching rates for {SYMBOL} from MT5...")

# Copy rates from terminal
h1_rates = mt5.copy_rates_range(SYMBOL, mt5.TIMEFRAME_H1, request_start, request_end)
m5_rates = mt5.copy_rates_range(SYMBOL, mt5.TIMEFRAME_M5, request_start, request_end)
m1_rates = mt5.copy_rates_range(SYMBOL, mt5.TIMEFRAME_M1, request_start, request_end)

# Process rates
rates_fetched = True
for name, rates in [("H1", h1_rates), ("M5", m5_rates), ("M1", m1_rates)]:
    if rates is None or len(rates) == 0:
        print(f"Error: Failed to retrieve {name} rates. Error code: {mt5.last_error()}")
        rates_fetched = False

if rates_fetched:
    # 1. Process H1
    df_h1 = pd.DataFrame(h1_rates)
    df_h1['time'] = pd.to_datetime(df_h1['time'], unit='s')
    # Filter: Keep target day, day before, and day after (00:00:00 of day-1 to 23:59:59 of day+1)
    df_h1 = df_h1[(df_h1['time'] >= pd.Timestamp(h1_start_local.year, h1_start_local.month, h1_start_local.day, 0, 0, 0)) &
                  (df_h1['time'] <= pd.Timestamp(h1_end_local.year, h1_end_local.month, h1_end_local.day, 23, 59, 59))]

    # 2. Process M5
    df_m5 = pd.DataFrame(m5_rates)
    df_m5['time'] = pd.to_datetime(df_m5['time'], unit='s')
    # Filter: Keep only target day (00:00:00 to 23:59:59)
    df_m5 = df_m5[(df_m5['time'] >= pd.Timestamp(target_datetime.year, target_datetime.month, target_datetime.day, 0, 0, 0)) &
                  (df_m5['time'] <= pd.Timestamp(target_datetime.year, target_datetime.month, target_datetime.day, 23, 59, 59))]

    # 3. Process M1
    df_m1 = pd.DataFrame(m1_rates)
    df_m1['time'] = pd.to_datetime(df_m1['time'], unit='s')
    # Filter: Keep only target day (00:00:00 to 23:59:59)
    df_m1 = df_m1[(df_m1['time'] >= pd.Timestamp(target_datetime.year, target_datetime.month, target_datetime.day, 0, 0, 0)) &
                  (df_m1['time'] <= pd.Timestamp(target_datetime.year, target_datetime.month, target_datetime.day, 23, 59, 59))]

    print("\nData fetched and filtered successfully!")
    print(f"H1 candles: {len(df_h1)}")
    print(f"M5 candles: {len(df_m5)}")
    print(f"M1 candles: {len(df_m1)}")
else:
    print("\nData retrieval failed. Please check symbol spelling and server connection.")

## Candlestick Charts Visualization

We render the three charts (H1, M5, M1) stacked vertically.

The charts are styled in **Quantower Style**:
- Background color is slate blue-gray (`#171e26`)
- Grid lines are set to subtle dark blue (`#222b35`) and placed in the background below the candles
- Open-to-Close bodies are rendered as thick flat bars in the foreground: bright green (`#3fb467`) for up moves and bright red (`#e75656`) for down moves
- High-to-Low wicks are rendered as **wide semi-transparent blocks** in the foreground, giving the chart a modern blocky structure
- **Y-Axis Tick Labels**: Automatically formatted to display standard decimal places based on the asset symbol (e.g. exactly 4 decimal places for major currencies like EURUSD, 3 for JPY, and 2 for Metals/Indices) for professional alignment

In [ ]:
if rates_fetched and len(df_h1) > 0 and (len(df_m5) > 0 or len(df_m1) > 0):
    bg_color = '#171e26'
    grid_color = '#222b35'
    text_color = '#7a869a'
    title_color = '#ffffff'

    # Determine decimal formatting dynamically based on symbol
    def get_decimal_places(sym, df):
        sym_upper = sym.upper()
        if "JPY" in sym_upper:
            return 3
        elif "XAU" in sym_upper or "XAG" in sym_upper:
            return 2
        elif any(x in sym_upper for x in ["USD", "EUR", "GBP", "AUD", "CAD", "NZD", "CHF"]):
            return 4
        else:
            if len(df) > 0:
                max_val = df['high'].max()
                if max_val > 1000:
                    return 2
                elif max_val > 100:
                    return 3
                else:
                    return 4
            return 4

    decimals = get_decimal_places(SYMBOL, df_h1)

    # Enable dark background style in matplotlib
    plt.style.use('dark_background')

    # Create the figure with 3 vertically stacked subplots
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 18), facecolor=bg_color)

    def draw_quantower_chart(ax, df, title, is_h1=False):
        # Set chart facecolor to match Quantower dark background
        ax.set_facecolor(bg_color)

        # Enable grid lines and force them to the background (below the candles)
        ax.grid(True, color=grid_color, linestyle='-', linewidth=0.5)
        ax.set_axisbelow(True)

        # Apply dynamic decimal place formatting on the Y-axis ticks
        ax.yaxis.set_major_formatter(ticker.FormatStrFormatter(f"%.{decimals}f"))

        if len(df) == 0:
            # Draw placeholder indicating no data
            ax.text(0.5, 0.5, "NO DATA AVAILABLE (WEEKEND/MARKET CLOSED)",
                    color='#ff5252', ha='center', va='center', fontsize=12, fontweight='bold', transform=ax.transAxes)
            ax.set_title(title, color=title_color, fontsize=12, pad=10, fontweight='semibold')
            return

        # Calculate candle width dynamically based on average time step
        times = mdates.date2num(df['time'])
        diffs = np.diff(times)
        # Use 70% of the median difference as the candle width
        width = np.median(diffs) * 0.7 if len(diffs) > 0 else 0.0005

        # Separate bullish, bearish, and doji candles
        up_mask = df['close'] > df['open']
        down_mask = df['close'] < df['open']
        flat_mask = df['close'] == df['open']

        # Quantower Colors
        bull_body = '#3fb467'     # Solid Green
        bull_shadow = '#3fb467'   # Translucent Green
        bear_body = '#e75656'     # Solid Red
        bear_shadow = '#e75656'   # Translucent Red
        flat_color = '#7a869a'    # Gray for flat/doji

        # 1. Draw High-to-Low ranges (wide shadows) with transparency (zorder=3 to place in foreground)
        if len(df.loc[up_mask]) > 0:
            ax.bar(df.loc[up_mask, 'time'], df.loc[up_mask, 'high'] - df.loc[up_mask, 'low'],
                   width=width, bottom=df.loc[up_mask, 'low'], color=bull_shadow, alpha=0.25, align='center', linewidth=0, zorder=3)

        if len(df.loc[down_mask]) > 0:
            ax.bar(df.loc[down_mask, 'time'], df.loc[down_mask, 'high'] - df.loc[down_mask, 'low'],
                   width=width, bottom=df.loc[down_mask, 'low'], color=bear_shadow, alpha=0.25, align='center', linewidth=0, zorder=3)

        if len(df.loc[flat_mask]) > 0:
            ax.bar(df.loc[flat_mask, 'time'], df.loc[flat_mask, 'high'] - df.loc[flat_mask, 'low'],
                   width=width, bottom=df.loc[flat_mask, 'low'], color=flat_color, alpha=0.25, align='center', linewidth=0, zorder=3)

        # 2. Draw Open-to-Close ranges (solid bodies) on top (zorder=4 to keep above shadows)
        if len(df.loc[up_mask]) > 0:
            ax.bar(df.loc[up_mask, 'time'], df.loc[up_mask, 'close'] - df.loc[up_mask, 'open'],
                   width=width, bottom=df.loc[up_mask, 'open'], color=bull_body, align='center', linewidth=0, zorder=4)

        if len(df.loc[down_mask]) > 0:
            ax.bar(df.loc[down_mask, 'time'], df.loc[down_mask, 'open'] - df.loc[down_mask, 'close'],
                   width=width, bottom=df.loc[down_mask, 'close'], color=bear_body, align='center', linewidth=0, zorder=4)

        if len(df.loc[flat_mask]) > 0:
            # Give flat body a small visual thickness
            flat_height = (df['high'].max() - df['low'].min()) * 0.005
            ax.bar(df.loc[flat_mask, 'time'], flat_height,
                   width=width, bottom=df.loc[flat_mask, 'open'] - flat_height/2, color=flat_color, align='center', linewidth=0, zorder=4)

        # Style spines
        for spine in ax.spines.values():
            spine.set_color(grid_color)

        # Customize tick labels and title colors
        ax.tick_params(colors=text_color, which='both')
        ax.yaxis.label.set_color(text_color)
        ax.xaxis.label.set_color(text_color)

        ax.set_title(title, color=title_color, fontsize=12, pad=10, fontweight='semibold')
        ax.set_ylabel("Price")

        # Format the x-axis depending on the timeframe
        if is_h1:
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
            ax.xaxis.set_major_locator(mdates.HourLocator(interval=6))
            ax.set_xlabel("Date and Time (MM-DD HH:MM)")
        else:
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
            ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))
            ax.set_xlabel("Time (HH:MM)")

        # Rotate dates for cleaner presentation
        plt.setp(ax.get_xticklabels(), rotation=30, horizontalalignment='right')

    # Draw top chart: H1 (Day before, day of, day after)
    h1_title = f"{SYMBOL} H1 Timeframe - Day Before to Day After ({h1_start_local.strftime('%Y-%m-%d')} to {h1_end_local.strftime('%Y-%m-%d')})"
    draw_quantower_chart(ax1, df_h1, h1_title, is_h1=True)

    # Draw middle chart: M5 (Intraday)
    m5_title = f"{SYMBOL} M5 Timeframe - Target Date ({TARGET_DATE})"
    draw_quantower_chart(ax2, df_m5, m5_title)

    # Draw bottom chart: M1 (Detailed Intraday)
    m1_title = f"{SYMBOL} M1 Timeframe - Target Date ({TARGET_DATE})"
    draw_quantower_chart(ax3, df_m1, m1_title)

    plt.tight_layout()
    plt.show()
else:
    print("Cannot plot charts because some or all timeframes contain no data.")
    print("Check if the date you selected was a weekend or if it is outside your broker's historical data limits.")

## Shutdown Connection

Always close the connection to the MetaTrader 5 terminal after completing data tasks.

In [ ]:
# Shutdown the connection
mt5.shutdown()
print("MetaTrader5 connection closed.")